In [ ]:
import json
from os import listdir as ls
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, ANALYSIS_TYPES
from emu_renewal.plotting import plot_analysis_specific_post
from emu_renewal.utils import get_countries_by_continent, split_list_into_segments, get_analysis_paths

set_matplotlib_formats("svg")

Need to make this handle both mob_exp and scale_floor for all the scaled analyses.

In [ ]:
import numpy as np
import arviz as az
from matplotlib import pyplot as plt
import pycountry
from emu_renewal.constants import MOB_SOURCE_COLOURS

In [ ]:
def plot_filtered_param_posts(countries, analysis_paths, relevant_analyses, param_name):
    n_cols = 4
    n_rows = int(np.ceil(len(countries) / n_cols))
    height = min(2.0 + n_rows * 2.0, 13.0)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=[12, height])
    flat_axes = axes.ravel()
    for c, iso3 in enumerate(countries):
        a_paths = analysis_paths[iso3]
        ax = flat_axes[c]
        avail_paths = a_paths.keys() & relevant_analyses
        for a in avail_paths:
            a_path = a_paths[a]
            idata = az.from_netcdf(a_path / "idata_filtered.nc")
            # Patch for parameter renaming between runs
            param_absent = param_name == "scale_exp" and "scale_exp" not in idata.posterior
            param = "mob_exp" if param_absent else param_name
            colour = MOB_SOURCE_COLOURS[a]
            az.plot_density(idata, ax=ax, hdi_prob=0.99, var_names=param, shade=0.2, colors=colour)
        ax.set_title(pycountry.countries.lookup(iso3).name)
    for c in range(c + 1, len(flat_axes)):
        flat_axes[c].set_axis_off()
    fig.tight_layout()
    return fig

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/inc_with_pol.json", "r"))
analysis_paths = get_analysis_paths(["49427755", "49507800", "49508160"], all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
relevant_analyses = [a for a in ANALYSIS_TYPES if a != "no_mob" and "fb_" not in a]
for cont, cont_countries in countries_by_cont.items():
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    for countries in split_list_into_segments(cont_countries, 16):
        display(plot_filtered_param_posts(countries, analysis_paths, relevant_analyses, "scale_floor"))
        display(plot_filtered_param_posts(countries, analysis_paths, relevant_analyses, "scale_exp"))